### 9 · Adaptive RAG — Self-Correcting Graph

Basic RAG retrieves once and generates. But what if the retrieval was bad?
What if the LLM hallucinated? Adaptive RAG adds **quality gates** that
detect failures and loop back to fix them.

### What you'll learn
```
1. Grading retrieved documents — are they relevant?
2. Hallucination detection — is the answer grounded in context?
3. Answer usefulness check — does it actually answer the question?
4. Conditional edges — retry or refine when quality is low
5. Query rewriting — when first retrieval fails, rephrase and try again
```

### Graph
```
START → retrieve → grade_documents ─┬─ relevant → generate → check_hallucination ─┬─ grounded → check_usefulness ─┬─ useful → END
                                     │                                              │                               │
                                     └─ not relevant → rewrite_query → retrieve     └─ hallucinated → generate      └─ not useful → rewrite_query
```

This is how production RAG systems work — they don't just hope for the best.

In [ ]:
from pathlib import Path
from typing import TypedDict, Literal

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import END, StateGraph
from dotenv import load_dotenv

load_dotenv(override=True)

VECTORSTORE_DIR = Path("../data/vectorstore")
EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vectorstore = FAISS.load_local(
    str(VECTORSTORE_DIR), embeddings, allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

print(f"Loaded {vectorstore.index.ntotal} vectors. Adaptive RAG ready.")

---
### Step 1 — Define State

State tracks the question, docs, answer, and a **retry counter** to prevent infinite loops.

In [ ]:
class GraphState(TypedDict):
    question: str                    # original or rewritten question
    original_question: str           # always the user's original question
    documents: list[Document]        # retrieved chunks
    filtered_documents: list[Document]  # chunks that passed grading
    answer: str                      # generated answer
    retries: int                     # retry counter (max 2)

MAX_RETRIES = 2

print(f"State defined. Max retries: {MAX_RETRIES}")

---
### Step 2 — Define All Nodes

Each node is a focused function with a single responsibility.

In [ ]:
import json as json_mod


def _parse_json(text: str) -> dict:
    """Parse JSON from LLM response."""
    cleaned = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json_mod.loads(cleaned)
    except json_mod.JSONDecodeError:
        return {}


# ── Node: Retrieve ────────────────────────────────────────────────────────────

def retrieve(state: GraphState) -> dict:
    """Retrieve documents for the current question."""
    docs = retriever.invoke(state["question"])
    retries = state.get("retries", 0)
    print(f"  [retrieve] → {len(docs)} docs (attempt {retries + 1})")
    return {"documents": docs}


# ── Node: Grade Documents ─────────────────────────────────────────────────────
# For each retrieved doc, the LLM decides: relevant or not?

GRADE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a relevance grader. Given a question and a document chunk, "
     "determine if the chunk contains information relevant to answering the question.\n\n"
     "Respond with ONLY a JSON object: {{\"relevant\": true or false}}"),
    ("human",
     "Question: {question}\n\nDocument:\n{document}"),
])

def grade_documents(state: GraphState) -> dict:
    """Grade each document for relevance. Keep only relevant ones."""
    chain = GRADE_PROMPT | llm | StrOutputParser()
    filtered = []

    for doc in state["documents"]:
        resp = chain.invoke({
            "question": state["question"],
            "document": doc.page_content[:500],
        })
        parsed = _parse_json(resp)
        if parsed.get("relevant", False):
            filtered.append(doc)

    print(f"  [grade] → {len(filtered)}/{len(state['documents'])} docs passed relevance check")
    return {"filtered_documents": filtered}


# ── Node: Generate Answer ─────────────────────────────────────────────────────

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant for the AMS Admin Tool team. "
     "Use ONLY the context below to answer. "
     "If the context doesn't contain enough information, say so.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])

def generate(state: GraphState) -> dict:
    """Generate answer from filtered documents."""
    context = "\n\n---\n\n".join(
        f"[{d.metadata.get('source', '?')}]\n{d.page_content}"
        for d in state["filtered_documents"]
    )
    chain = ANSWER_PROMPT | llm | StrOutputParser()
    answer = chain.invoke({
        "question": state["original_question"],
        "context": context,
    })
    print(f"  [generate] → {len(answer)} chars")
    return {"answer": answer}


# ── Node: Check Hallucination ─────────────────────────────────────────────────
# Is the answer grounded in the retrieved context?

HALLUCINATION_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a hallucination detector. Given a context and an answer, "
     "determine if the answer is FULLY supported by the context.\n\n"
     "Respond with ONLY a JSON object: {{\"grounded\": true or false, \"reason\": \"brief explanation\"}}"),
    ("human",
     "Context:\n{context}\n\nAnswer:\n{answer}"),
])

def check_hallucination(state: GraphState) -> dict:
    """Check if the answer is grounded in retrieved context."""
    context = "\n\n".join(d.page_content for d in state["filtered_documents"])
    chain = HALLUCINATION_PROMPT | llm | StrOutputParser()
    resp = chain.invoke({"context": context, "answer": state["answer"]})
    parsed = _parse_json(resp)
    grounded = parsed.get("grounded", True)
    print(f"  [hallucination_check] → {'GROUNDED' if grounded else 'HALLUCINATED'}: {parsed.get('reason', '')}")
    return {}  # decision is made in the conditional edge


# ── Node: Check Usefulness ────────────────────────────────────────────────────
# Does the answer actually address the question?

USEFULNESS_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a quality checker. Given a question and an answer, "
     "determine if the answer is useful and resolves the question.\n\n"
     "Respond with ONLY a JSON object: {{\"useful\": true or false, \"reason\": \"brief explanation\"}}"),
    ("human",
     "Question:\n{question}\n\nAnswer:\n{answer}"),
])

def check_usefulness(state: GraphState) -> dict:
    """Check if the answer is useful for the question."""
    chain = USEFULNESS_PROMPT | llm | StrOutputParser()
    resp = chain.invoke({"question": state["original_question"], "answer": state["answer"]})
    parsed = _parse_json(resp)
    useful = parsed.get("useful", True)
    print(f"  [usefulness_check] → {'USEFUL' if useful else 'NOT USEFUL'}: {parsed.get('reason', '')}")
    return {}


# ── Node: Rewrite Query ───────────────────────────────────────────────────────
# When retrieval or answer quality is poor, rephrase the question.

REWRITE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a query rewriter. The original query didn't retrieve good results. "
     "Rewrite it to be more specific and likely to match relevant documents.\n\n"
     "Output ONLY the rewritten query."),
    ("human", "Original query: {question}"),
])

def rewrite_query(state: GraphState) -> dict:
    """Rewrite the question for better retrieval."""
    chain = REWRITE_PROMPT | llm | StrOutputParser()
    new_question = chain.invoke({"question": state["question"]})
    retries = state.get("retries", 0) + 1
    print(f"  [rewrite] '{state['question']}' → '{new_question}' (retry {retries})")
    return {"question": new_question, "retries": retries}


print("All nodes defined.")

---
### Step 3 — Conditional Edge Functions

These functions decide which path to take at decision points.
They re-run the check logic (lightweight) and return the next node name.

In [ ]:
def decide_after_grading(state: GraphState) -> Literal["generate", "rewrite_query"]:
    """After grading: generate if enough relevant docs, else rewrite."""
    if len(state.get("filtered_documents", [])) >= 2:
        return "generate"

    if state.get("retries", 0) >= MAX_RETRIES:
        print(f"  [decision] not enough docs but max retries reached, generating anyway")
        return "generate"

    print(f"  [decision] only {len(state.get('filtered_documents', []))} relevant docs, rewriting query")
    return "rewrite_query"


def decide_after_hallucination_check(state: GraphState) -> Literal["check_usefulness", "generate"]:
    """After hallucination check: proceed if grounded, regenerate if not."""
    context = "\n\n".join(d.page_content for d in state["filtered_documents"])
    chain = HALLUCINATION_PROMPT | llm | StrOutputParser()
    resp = chain.invoke({"context": context, "answer": state["answer"]})
    parsed = _parse_json(resp)

    if parsed.get("grounded", True):
        return "check_usefulness"

    if state.get("retries", 0) >= MAX_RETRIES:
        print(f"  [decision] hallucinated but max retries reached")
        return "check_usefulness"

    print(f"  [decision] hallucination detected, regenerating")
    return "generate"


def decide_after_usefulness_check(state: GraphState) -> Literal["end", "rewrite_query"]:
    """After usefulness check: finish if useful, rewrite if not."""
    chain = USEFULNESS_PROMPT | llm | StrOutputParser()
    resp = chain.invoke({"question": state["original_question"], "answer": state["answer"]})
    parsed = _parse_json(resp)

    if parsed.get("useful", True):
        return "end"

    if state.get("retries", 0) >= MAX_RETRIES:
        print(f"  [decision] not useful but max retries reached")
        return "end"

    print(f"  [decision] answer not useful, rewriting query")
    return "rewrite_query"


print("Conditional edge functions defined.")

---
### Step 4 — Build the Adaptive Graph

Wire everything together with conditional edges.

In [ ]:
workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate", generate)
workflow.add_node("check_hallucination", check_hallucination)
workflow.add_node("check_usefulness", check_usefulness)
workflow.add_node("rewrite_query", rewrite_query)

# Entry
workflow.set_entry_point("retrieve")

# Fixed edges
workflow.add_edge("retrieve", "grade_documents")
workflow.add_edge("rewrite_query", "retrieve")

# Conditional edges
workflow.add_conditional_edges(
    "grade_documents",
    decide_after_grading,
    {"generate": "generate", "rewrite_query": "rewrite_query"},
)

workflow.add_conditional_edges(
    "check_hallucination",
    decide_after_hallucination_check,
    {"check_usefulness": "check_usefulness", "generate": "generate"},
)

workflow.add_conditional_edges(
    "check_usefulness",
    decide_after_usefulness_check,
    {"end": END, "rewrite_query": "rewrite_query"},
)

workflow.add_edge("generate", "check_hallucination")

adaptive_graph = workflow.compile()
print("Adaptive RAG graph compiled.")

In [ ]:
from IPython.display import Image, display
display(Image(adaptive_graph.get_graph().draw_mermaid_png()))

---
### Step 5 — Test: Normal Query (should pass all checks)

A well-scoped question that our docs can answer.

In [ ]:
question = "What state management libraries does the AMS Admin Tool use?"
print(f"Question: {question}\n")

result = adaptive_graph.invoke({
    "question": question,
    "original_question": question,
    "retries": 0,
})

print(f"\n{'='*70}")
print(f"Retries used: {result.get('retries', 0)}")
print(f"Filtered docs: {len(result.get('filtered_documents', []))}")
print(f"\nANSWER:\n{result['answer']}")

---
### Step 6 — Test: Vague Query (may trigger rewrite)

In [ ]:
question = "How do I make my code better?"
print(f"Question: {question}\n")

result = adaptive_graph.invoke({
    "question": question,
    "original_question": question,
    "retries": 0,
})

print(f"\n{'='*70}")
print(f"Retries used: {result.get('retries', 0)}")
print(f"Final question: {result['question']}")
print(f"Filtered docs: {len(result.get('filtered_documents', []))}")
print(f"\nANSWER:\n{result['answer']}")

---
### Step 7 — Test: Out-of-Domain Query (should hit retry limit)

In [ ]:
question = "What is the weather like in Mumbai?"
print(f"Question: {question}\n")

result = adaptive_graph.invoke({
    "question": question,
    "original_question": question,
    "retries": 0,
})

print(f"\n{'='*70}")
print(f"Retries used: {result.get('retries', 0)}")
print(f"Final question: {result['question']}")
print(f"Filtered docs: {len(result.get('filtered_documents', []))}")
print(f"\nANSWER:\n{result['answer']}")

---
### Step 8 — Run All Test Queries & Summary

In [ ]:
TEST_QUERIES = [
    "What state management libraries does the AMS Admin Tool use?",
    "How does the journey canvas work?",
    "What are the anti-patterns to avoid?",
    "How do I make my code better?",
    "What is the capital of France?",
]

print(f"{'Question':<55} {'Retries':>8} {'Docs':>5} {'Answer (first 80 chars)'}")
print("-" * 160)

for q in TEST_QUERIES:
    result = adaptive_graph.invoke({
        "question": q,
        "original_question": q,
        "retries": 0,
    })
    retries = result.get("retries", 0)
    n_docs = len(result.get("filtered_documents", []))
    answer_preview = result["answer"][:80].replace("\n", " ")
    print(f"{q[:53]:<55} {retries:>8} {n_docs:>5} {answer_preview}")

---
### Takeaways

| Concept | What you learned |
|---------|------------------|
| **Document grading** | LLM scores each chunk for relevance before generation |
| **Hallucination detection** | Post-generation check ensures answer is grounded in context |
| **Usefulness check** | Verifies the answer actually addresses the question |
| **Query rewriting** | When retrieval fails, rephrase and retry |
| **Conditional edges** | LangGraph routes to different nodes based on quality checks |
| **Retry budget** | `MAX_RETRIES` prevents infinite loops |

**This is the pattern used in production RAG systems:**
1. Retrieve → Grade → Generate → Verify → Deliver
2. At each stage, if quality is low, loop back and fix
3. Retry budget ensures the system always terminates

**Combine with previous notebooks for maximum quality:**
- Hybrid search (notebook 8) for better initial retrieval
- Reranking (notebook 7) after retrieval, before grading
- Memory (notebook 6) for multi-turn conversations
- Evaluation (notebook 5) to measure the improvement